Understanding the dataset.

In [7]:
import pandas as pd
df = pd.read_csv('../data/raw/sales_data.csv')
print(df.shape)

(76000, 16)


In [8]:
print(df['Category'].unique())

<StringArray>
['Electronics', 'Clothing', 'Groceries', 'Toys', 'Furniture']
Length: 5, dtype: str


the categories in the cvs , which will be filtered to only to remain with clothing specific for this project.

In [9]:
print(df[['Units Sold', 'Demand']].describe())


         Units Sold        Demand
count  76000.000000  76000.000000
mean      88.827316    104.317158
std       43.994525     46.964801
min        0.000000      4.000000
25%       58.000000     71.000000
50%       84.000000    100.000000
75%      114.000000    133.000000
max      426.000000    430.000000


Demand is consistently higher than Units Sold .On average, by about 15-16 units. Demand never goes below 4, but Units Sold goes all the way down to 0.

In [10]:
print((df['Units Sold'] != df['Demand']).sum())

74440


Out of 76,000 rows, 74,440 of them (98%) have Units Sold not equal to Demand.
hypotheses; 
    Demand = what customers actually wanted to buy that day (the true, unconstrained signal)
    Units Sold = what was actually recorded as sold, which gets capped when there isn't enough stock to meet that demand

In [11]:
print(df[df['Units Sold'] < df['Demand']][['Inventory Level', 'Units Sold', 'Demand']].head(10))

    Inventory Level  Units Sold  Demand
0               195         102     115
1               117         117     229
2               247         114     157
3               139          45      52
6               118          81      94
7               244          42      61
8               115          88     129
10              215          51      64
13              252         121     165
14              119          55      64


Units Sold exactly equals Inventory Level. That's a strong stockout signal. on row 1.
But Row 0: Inventory Level: 195, Units Sold: 102, Demand: 115 here Inventory (195) is higher than Demand (115), yet Units Sold (102) is still below both. If it were purely "sold = whatever was available, capped by demand," this row shouldn't exist there was enough stock to meet demand, so why wasn't it sold?

In [12]:
predicted_sold = df[['Inventory Level', 'Demand']].min(axis=1)
matches = (df['Units Sold'] == predicted_sold).sum()
print(f"{matches} out of {len(df)} rows match the 'sold = min(inventory, demand)' pattern")
print(df[df['Units Sold'] != predicted_sold][['Inventory Level', 'Units Sold', 'Demand']].head(10))

9408 out of 76000 rows match the 'sold = min(inventory, demand)' pattern
    Inventory Level  Units Sold  Demand
0               195         102     115
2               247         114     157
3               139          45      52
4               152          65      59
5               209          60      55
6               118          81      94
7               244          42      61
8               115          88     129
9               192          70      69
10              215          51      64


Only 9,408 out of 76,000 (about 12%) fit the simple "sold = whichever is smaller" rule. That means my hypothesis was too simple stockouts alone don't explain the gap.
For example row 4 specifically: Inventory 152, Units Sold 65, Demand 59. Here, Demand (59) is actually less than Inventory (152) there was more than enough stock to satisfy every unit of demand. Yet Units Sold (65) doesn't even match Demand cleanly either. If it were purely a stock availability story, this row shouldn't look like this at all.
Went back to the cvs and noted demand is a prediction estimate, not what actually happened. While units sold is the real thing that happened.

This connects to my field research ; vendor E2, she ran out of navy hijabs during back-to-school season. If i trained SokoSmart's model on Units Sold, the model would only ever see "sold: whatever was left" it would never learn that true demand was actually higher. It would systematically underforecast, telling vendors to reorder less than they actually need which recreates the exact stockout problem the whole dissertation is trying to solve.

In [13]:
# Check the date range of the full dataset
df['Date'] = pd.to_datetime(df['Date'])
print("Earliest date:", df['Date'].min())
print("Latest date:", df['Date'].max())
print("Total days covered:", (df['Date'].max() - df['Date'].min()).days)

Earliest date: 2022-01-01 00:00:00
Latest date: 2024-01-30 00:00:00
Total days covered: 759


Full dataset spans 2022-01-01 to 2024-01-30; about 759 days, roughly 2 years. Solid amount of history for ARIMA/LSTM, which both benefit from multiple seasonal cycles to learn from.

In [14]:
# Filter down to Clothing only, since that's SokoSmart's actual domain
clothing_df = df[df['Category'] == 'Clothing']
print("\nRows after filtering to Clothing:", clothing_df.shape[0])
print("Unique products:", clothing_df['Product ID'].nunique())
print("Unique stores:", clothing_df['Store ID'].nunique())
print("Unique regions:", clothing_df['Region'].unique())


Rows after filtering to Clothing: 12160
Unique products: 13
Unique stores: 5
Unique regions: <StringArray>
['North', 'South', 'East', 'West']
Length: 4, dtype: str


Clothing filter leaves 12,160 rows, across 13 products and 5 stores, spread over 4 regions (North/South/East/West).

But something doesn't add up; 13 products × 5 stores = 65 possible product-store combinations. If you divide 12,160 rows by 65, you get about 187 rows per combination but the full dataset spans 759 days. That's a big gap: 187 is nowhere close to 759.
this matters because ARIMA and LSTM in particular expect a roughly continuous, evenly-spaced time series. A choppy one with gaps needs to be handled deliberately (interpolation, resampling, or restricting to combinations with enough continuous history), not ignored.

In [15]:
# How many rows does each product store combination actually have?
combo_counts = clothing_df.groupby(['Store ID', 'Product ID']).size()
print(combo_counts.describe())
print("\nSmallest combos:\n", combo_counts.sort_values().head())

count     16.0
mean     760.0
std        0.0
min      760.0
25%      760.0
50%      760.0
75%      760.0
max      760.0
dtype: float64

Smallest combos:
 Store ID  Product ID
S001      P0002         760
          P0003         760
          P0009         760
S002      P0006         760
          P0015         760
dtype: int64


mean: 760.0, std: 0.0, min: 760.0, max: 760.0 — every single one of those 16 combinations has exactly 760 rows, with zero variation. That's actually very clean: 12,160 ÷ 16 = 760 exactly, matching the 760-day date range almost perfectly. This strongly suggests each existing combination has one row for every single day, with no missing days.

In [16]:
# Are dates roughly continuous for one combination, or full of gaps?
sample = clothing_df[(clothing_df['Store ID'] == clothing_df['Store ID'].iloc[0]) &
                      (clothing_df['Product ID'] == clothing_df['Product ID'].iloc[0])].sort_values('Date')
print("\nSample combo date range:", sample['Date'].min(), "to", sample['Date'].max())
print("Number of rows in this combo:", len(sample))
print("Days between first and last:", (sample['Date'].max() - sample['Date'].min()).days)


Sample combo date range: 2022-01-01 00:00:00 to 2024-01-30 00:00:00
Number of rows in this combo: 760
Days between first and last: 759


this is about as clean as data gets. Here's the check: it has 760 rows with zero gaps or duplicates, the gap between the first and last date should be exactly 759 days (760 points create 759 gaps between them, like fence posts and fence sections). That's exactly what gotten. This one combination has a perfect, unbroken daily record for the full 2 year span, no missing days, no duplicate days.